# `get_q()`

`nematics3d.get_q()` constructs symmetric, traceless $Q$-tensor arrays from scalar-order and director data.

## What `get_q()` is for

Use `get_q()` when a configuration is expressed through a director $\mathbf{n}$ and scalar order $S$, but a later calculation or `Nematics3D` object expects $Q$-tensor data. This commonly occurs while preparing analytic initial conditions, building a field from a known director texture, or reconstructing $Q$ after editing its eigensystem.

In one call, the function:

- validates and normalizes director data;
- broadcasts director and scalar-order fields to a common shape;
- constructs either the uniaxial or biaxial tensor convention; and
- returns a `NumPy` array whose last two axes contain each $3\times3$ tensor.

### Uniaxial convention

The uniaxial definition is

$$Q=S\left(\mathbf{n}\mathbf{n}-\frac{I}{3}\right).$$

For a unit director, the eigenvalue along $\mathbf{n}$ is $2S/3$ and both transverse eigenvalues are $-S/3$. Their sum is zero, which makes $Q$ traceless.

### Biaxial convention

The optional biaxial term is

$$P(\mathbf{m}\mathbf{m}-\mathbf{l}\mathbf{l}),\qquad \mathbf{l}=\mathbf{n}\times\mathbf{m}.$$

The three eigenvalues associated with $\mathbf{n}$, $\mathbf{m}$, and $\mathbf{l}$ are respectively $2S/3$, $-S/3+P$, and $-S/3-P$. `get_q()` allows signed $P$ and does not impose a physical range on either scalar-order parameter.

## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The next cell only imports `NumPy` and the public `nematics3d` package.

In [1]:
import numpy as np
import nematics3d as n3d
from nematics3d.datatypes import as_qfield9

## Minimal example

The shortest useful call needs only a director:

- `n=[1, 0, 0]` selects the $x$ direction;
- omitting `S` uses the default $S=1$.

In [2]:
Q = n3d.get_q([1.0, 0.0, 0.0])
print(Q)

[[ 0.66666667  0.          0.        ]
 [ 0.         -0.33333333  0.        ]
 [ 0.          0.         -0.33333333]]


The returned matrix is diagonal because the supplied director follows a coordinate axis. Its eigenvalue along $\mathbf{n}$ is $2S/3$, while the two transverse eigenvalues are both $-S/3$. The entries sum to zero, so the tensor is traceless.

### What is returned

By default, returns `QField9` (a semantic alias of `numpy.ndarray`) with final axes `(3, 3)`. Set `output="q5"` to return the five independent components instead. Any earlier axes are the broadcast field or batch dimensions. Either representation can be passed to Nematics3D interfaces that accept $Q$-tensor data.

## Arguments

```python
nematics3d.get_q(
    n,
    S=None,
    *,
    m=None,
    P=None,
    output="q9",
)
```

| Argument | What it controls | Typical form |
| --- | --- | --- |
| `n` | Primary director $\mathbf{n}$; normalized point by point | shape `(..., 3)` |
| `S` | Uniaxial scalar order $S$; defaults to 1 | scalar or broadcastable array |
| `m` | Secondary director $\mathbf{m}$ for biaxial construction | shape `(..., 3)` or `None` |
| `P` | Signed biaxial order $P$ | scalar, broadcastable array, or `None` |
| `output` | Output representation; `"q9"` is the full tensor and `"q5"` stores only independent components | `"q9"` or `"q5"` |

`m` and `P` are a pair: provide both for biaxial construction or omit both for uniaxial construction. The `output` value is checked against the two supported strings, so other spellings raise an error.

### Example: choosing Q5 or Q9 output

`output="q9"` returns the full symmetric, traceless $3\times3$ tensor and remains the default. `output="q5"` returns its five independent components in the order `(Qxx, Qxy, Qxz, Qyy, Qyz)`. The omitted entries follow from symmetry and $Q_{zz}=-Q_{xx}-Q_{yy}$, so Q5 contains the same physical information while using less memory.

The Q5 result is constructed directly; `get_q()` does not first allocate a Q9 array.

In [3]:
Q9 = n3d.get_q([1.0, 0.0, 0.0], S=0.6, output="q9")
Q5 = n3d.get_q([1.0, 0.0, 0.0], S=0.6, output="q5")
print("Q9 shape:", Q9.shape)
print("Q5 shape:", Q5.shape)
print("same tensor:", np.allclose(as_qfield9(Q5, is_strict_3d_field=False), Q9))

Q9 shape: (3, 3)
Q5 shape: (5,)
same tensor: True


### Example: choosing the scalar order $S$

Changing `S` scales all three uniaxial eigenvalues without changing the director. Here the input director is not normalized; `get_q()` normalizes it before constructing $Q$.

In [4]:
Q_ordered = n3d.get_q([2.0, 0.0, 0.0], S=0.6)
print(Q_ordered)
print("eigenvalues:", np.linalg.eigvalsh(Q_ordered)[::-1])

[[ 0.4  0.   0. ]
 [ 0.  -0.2  0. ]
 [ 0.   0.  -0.2]]
eigenvalues: [ 0.4 -0.2 -0.2]


### Example: constructing a field or batch

Director and scalar-order leading dimensions follow ordinary `NumPy` broadcasting. In this example, `directors` stores two directors and `scalar_order` supplies one value for each.

In [5]:
directors = np.array(
    [
        [1.0, 0.0, 0.0],
        [0.0, 1.0, 0.0],
    ]
)
scalar_order = np.array([0.6, 0.3])
Q_batch = n3d.get_q(directors, S=scalar_order)
print("input director shape:", directors.shape)
print("output tensor shape:", Q_batch.shape)

input director shape: (2, 3)
output tensor shape: (2, 3, 3)


### Example: adding biaxial order

For biaxial data, `m` supplies a unit direction orthogonal to $\mathbf{n}$. The function derives the third direction $\mathbf{l}=\mathbf{n}\times\mathbf{m}$. A positive `P` raises the eigenvalue along $\mathbf{m}$ and lowers it along $\mathbf{l}$; a negative value reverses that assignment.

In [6]:
n = np.array([1.0, 0.0, 0.0])
m = np.array([0.0, 1.0, 0.0])
Q_biaxial = n3d.get_q(n, S=0.6, m=m, P=0.2)
print(Q_biaxial)
print("eigenvalues:", np.linalg.eigvalsh(Q_biaxial)[::-1])

[[ 4.00000000e-01  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  2.77555756e-17  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00 -4.00000000e-01]]
eigenvalues: [ 4.00000000e-01  2.77555756e-17 -4.00000000e-01]


## Special examples

### The isotropic case

An isotropic tensor is represented by $S=0$. A valid director is still required so malformed or missing orientation data is not silently accepted; its direction has no effect after multiplication by zero.

In [7]:
Q_isotropic = n3d.get_q([1.0, 0.0, 0.0], S=0.0)
print(Q_isotropic)

[[ 0.  0.  0.]
 [ 0. -0.  0.]
 [ 0.  0. -0.]]


### Director-sign invariance

Nematic directors are headless: $\mathbf{n}$ and $-\mathbf{n}$ describe the same axis. Because `get_q()` uses dyadic products, reversing either director does not change $Q$.

In [8]:
Q_positive = n3d.get_q(n, S=0.6, m=m, P=0.2)
Q_negative = n3d.get_q(-n, S=0.6, m=-m, P=0.2)
print(np.allclose(Q_positive, Q_negative))

True


## Details

### Broadcasting and output shape

The leading shapes of `n`, `S`, `m`, and `P` follow `NumPy` broadcasting. The output leading shape is their common broadcast shape, followed by `(3, 3)` for `output="q9"` or `(5,)` for `output="q5"`.

## Round trip through `q_diagonalize()`

When $\mathbf{n}$ remains the dominant eigenvector and $P\geq0$, the complete eigensystem orders $\mathbf{m}$ before $\mathbf{l}$. The original biaxial order is then half the difference between the two transverse eigenvalues. Director signs may change during diagonalization, but the reconstructed $Q$ is unchanged.

In [9]:
result = n3d.q_diagonalize(Q_biaxial, is_biaxial=True)
P_recovered = (result.eigenvalues[1] - result.eigenvalues[2]) / 2
Q_reconstructed = n3d.get_q(
    result.n,
    S=result.S,
    m=result.eigenvectors[:, 1],
    P=P_recovered,
)
print(np.allclose(Q_reconstructed, Q_biaxial))

True


## Possible issues

### A zero-director error is raised

At least one director has zero or near-zero norm. Supply a valid direction even where $S=0$; the isotropic example above shows this pattern.

### `m` and `P` must be supplied together

Only one part of the biaxial specification was provided. Supply both arguments, or omit both to construct a uniaxial tensor.

### The directors are reported as non-orthogonal

The supplied $\mathbf{m}$ is not perpendicular to $\mathbf{n}$ at one or more field points. Correct the director data before calling `get_q()`; the function deliberately does not project it silently.

### The leading shapes do not broadcast

The field or batch dimensions of the supplied arguments are incompatible under `NumPy` broadcasting. Inspect the input shapes and add singleton dimensions where a value should repeat along an axis.

## Where `get_q()` is used

- `QFieldObject` uses `get_q()` when it receives $S$ and $\mathbf{n}$ instead of a preconstructed $Q$ field.
- Principal-plane analysis constructs local $Q$ tensors from sampled directors before averaging and diagonalizing them.

## Useful Links

### Referenced in this tutorial

- [`q_diagonalize()`](../analysis/q_diagonalization/q_diagonalize.ipynb) — converts $Q$ into scalar order, a principal director, and optionally the complete eigensystem.